# Strategy Selection [Step 2 - Routing Retrieval Strategies]

> **MLCourse - Agentic AI - Adaptive RAG**

Once a query is classified, the system must decide *how* to retrieve
relevant information.  A factual lookup about a specific character calls
for keyword (BM25) search; a thematic question benefits from semantic
vector search; a conversational greeting needs no retrieval at all; and
an out-of-scope query should be redirected.  This notebook builds a
router that takes a classification label and picks one of four strategies:
vector search, keyword search, web search, or direct answer.

### What you will learn

1. How to implement four distinct retrieval strategies behind one interface.
2. How a routing function maps classification labels to strategies.
3. When each strategy is the best fit for a given query type.
4. How to log routing decisions for inspection and debugging.

In [1]:
import os
import re
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

False

### 1. Configuration


In [ ]:
def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until the track directory appears."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(f"Could not find '{target}' above {start}")

TRACK = find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(exist_ok=True)

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)


### 2. Ingest Alice and Build Both Retrievers


In [ ]:
# We create a vector retriever (FAISS) and a BM25 keyword retriever from
# the same document set so the router can dispatch to either one.

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

ALICE_PATH = DATA / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")[:20_000]

marks = list(re.finditer(r"^CHAPTER [IVX]+\.", raw_text, flags=re.MULTILINE))
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

documents = []
for i, mark in enumerate(marks):
    seg_end = marks[i + 1].start() if i + 1 < len(marks) else len(raw_text)
    for piece in splitter.split_text(raw_text[mark.start():seg_end]):
        documents.append(Document(
            page_content=piece,
            metadata={"source": "alice", "chapter": str(i + 1)},
        ))

print(f"[ingest] chunks: {len(documents)}")


### 2a. Vector Retriever (FAISS)


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(documents, embeddings)
vector_retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print("[retriever] FAISS vector retriever ready (k=4)")


### 2b. Keyword Retriever (BM25)


In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents, k=4)
print("[retriever] BM25 keyword retriever ready (k=4)")


### 3. Initialize the LLM


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("[llm] ChatOllama ready:", llm.model)


### 4. Define the State


In [ ]:
# The state carries the query, the classification label, the chosen
# strategy, retrieved context, final answer, and a routing log.

from typing import TypedDict

class RoutingState(TypedDict):
    query: str                       # original user question
    classification: str              # "factual", "analytical", "conversational", "out_of_scope"
    strategy: str                    # "vector", "bm25", "web", "direct"
    context: str                     # retrieved text joined together
    answer: str                      # final generated answer
    route_log: list                  # audit trail of routing decisions

print("[state] RoutingState defined")


### 5. Define the Four Strategies


In [ ]:
# Each strategy is implemented as a function that returns retrieved context
# or a direct answer.  The web search strategy is simulated here with a
# stub that falls back to the LLM -- in production you would wire it to
# a real search API.

def strategy_vector(query: str) -> str:
    """Retrieve chunks via semantic vector similarity search."""
    docs = vector_retriever.invoke(query)
    context = "\n\n".join(d.page_content for d in docs)
    print(f"  [vector] retrieved {len(docs)} chunks ({len(context)} chars)")
    return context

def strategy_bm25(query: str) -> str:
    """Retrieve chunks via BM25 keyword matching."""
    docs = bm25_retriever.invoke(query)
    context = "\n\n".join(d.page_content for d in docs)
    print(f"  [bm25] retrieved {len(docs)} chunks ({len(context)} chars)")
    return context

def strategy_web(query: str) -> str:
    """Simulated web search -- in production, call a search API."""
    # For offline/safe execution we use a stub that tells the LLM
    # to answer from general knowledge as if it had searched the web.
    print(f"  [web] simulated web search for: {query[:60]}...")
    return "[Web search not available in local mode. Use LLM knowledge.]"

def strategy_direct(query: str) -> str:
    """No retrieval; the LLM answers from parametric memory."""
    print(f"  [direct] no retrieval needed")
    return ""

STRATEGY_FUNCTIONS = {
    "vector": strategy_vector,
    "bm25": strategy_bm25,
    "web": strategy_web,
    "direct": strategy_direct,
}

print("[strategies] 4 strategies registered: vector, bm25, web, direct")


### 6. Build the Strategy Router


In [ ]:
# The router function maps a classification label to a retrieval strategy.
# The mapping encodes domain knowledge about which strategy works best
# for each query type.

STRATEGY_MAP = {
    "factual":        "bm25",       # exact keyword match for specific details
    "analytical":     "vector",     # semantic search for conceptual questions
    "conversational": "direct",     # no retrieval needed
    "out_of_scope":   "web",        # attempt external search or redirect
}

def route_by_classification(state: RoutingState) -> str:
    """Map a classification label to a retrieval strategy.

    Args:
        state: current graph state with 'classification' key.

    Returns:
        The strategy label string.
    """
    classification = state.get("classification", "factual")
    strategy = STRATEGY_MAP.get(classification, "vector")
    print(f"  [router] {classification} -> {strategy}")
    return strategy

print("[router] Strategy map:")
for cls, strat in STRATEGY_MAP.items():
    print(f"  {cls:20s} -> {strat}")


### 7. Classification Node


In [ ]:
# Uses the same criteria from notebook 01 to classify the query before
# routing.

from langchain_core.prompts import ChatPromptTemplate

CLASSIFICATION_CRITERIA = """
You are a query classifier for an Alice in Wonderland knowledge base.
Classify the user query into EXACTLY one category:

- 'factual': asks about specific events, characters, dialogue, or details
  from Alice in Wonderland.
- 'analytical': requires reasoning about themes, causes, comparisons, or
  interpretations.
- 'conversational': greetings, opinions, meta questions, or small talk.
- 'out_of_scope': completely unrelated to Alice in Wonderland.

Reply with ONLY the category label in lowercase.
"""

classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", CLASSIFICATION_CRITERIA),
    ("user", "{query}")
])

VALID_CATEGORIES = ("factual", "analytical", "conversational", "out_of_scope")

def classify(state: RoutingState) -> dict:
    """Classify the incoming query into one of four categories."""
    response = (classifier_prompt | llm).invoke({"query": state["query"]})
    raw = response.content.strip().lower()
    category = "out_of_scope"
    for cat in VALID_CATEGORIES:
        if cat in raw:
            category = cat
            break
    log = state.get("route_log", []) + [f"classify -> {category}"]
    print(f"  [classify] '{state['query'][:50]}...' -> {category}")
    return {"classification": category, "route_log": log}


### 8. Retrieval Nodes


In [ ]:
# One node per strategy.  Each reads the query, calls the strategy function,
# and updates the state with the context.

def retrieve_vector(state: RoutingState) -> dict:
    """Node: retrieve via vector similarity."""
    context = strategy_vector(state["query"])
    log = state.get("route_log", []) + ["retrieved via vector"]
    return {"context": context, "route_log": log}

def retrieve_bm25(state: RoutingState) -> dict:
    """Node: retrieve via BM25 keyword search."""
    context = strategy_bm25(state["query"])
    log = state.get("route_log", []) + ["retrieved via bm25"]
    return {"context": context, "route_log": log}

def retrieve_web(state: RoutingState) -> dict:
    """Node: retrieve via simulated web search."""
    context = strategy_web(state["query"])
    log = state.get("route_log", []) + ["retrieved via web (simulated)"]
    return {"context": context, "route_log": log}

def answer_direct(state: RoutingState) -> dict:
    """Node: answer from LLM knowledge without retrieval."""
    from langchain_core.prompts import ChatPromptTemplate as CPT
    prompt = CPT.from_messages([
        ("system",
         "Answer the question concisely in 1-3 sentences using your own "
         "knowledge.  Be helpful and friendly."),
        ("user", "{query}")
    ])
    response = (prompt | llm).invoke({"query": state["query"]})
    log = state.get("route_log", []) + ["direct answer (no retrieval)"]
    print(f"  [direct] answered from LLM knowledge")
    return {"answer": response.content, "context": "", "route_log": log}


### 9. Generate Answer Node


In [ ]:
# Shared by the three retrieval branches (vector, bm25, web) to produce
# a grounded answer from the retrieved context.

def generate_answer(state: RoutingState) -> dict:
    """Generate an answer grounded in the retrieved context."""
    from langchain_core.prompts import ChatPromptTemplate as CPT
    prompt = CPT.from_messages([
        ("system",
         "Answer the question using ONLY the provided context. "
         "If the context does not contain enough information, say so. "
         "Keep the answer to 2-4 sentences."),
        ("user", "Question: {query}\n\nContext:\n{context}")
    ])
    response = (prompt | llm).invoke({
        "query": state["query"],
        "context": state["context"]
    })
    log = state.get("route_log", []) + [f"generated answer ({len(response.content)} chars)"]
    print(f"  [generate] answer length: {len(response.content)} chars")
    return {"answer": response.content, "route_log": log}


### 10. Build the Strategy-Selection Graph


In [ ]:
# Flow: classify -> route -> (vector | bm25 | web | direct) -> generate -> END
# The direct branch skips the generator since it already produced the answer.

from langgraph.graph import StateGraph, START, END

graph = StateGraph(RoutingState)

# Nodes
graph.add_node("classify", classify)
graph.add_node("retrieve_vector", retrieve_vector)
graph.add_node("retrieve_bm25", retrieve_bm25)
graph.add_node("retrieve_web", retrieve_web)
graph.add_node("direct_answer", answer_direct)
graph.add_node("generate", generate_answer)

# Entry
graph.add_edge(START, "classify")

# Classification determines the next node.
def route_after_classify(state: RoutingState) -> str:
    """Dispatch to the appropriate retrieval strategy node."""
    strategy = route_by_classification(state)
    return f"retrieve_{strategy}" if strategy != "direct" else "direct_answer"

graph.add_conditional_edges(
    "classify",
    route_after_classify,
    {
        "retrieve_vector": "retrieve_vector",
        "retrieve_bm25": "retrieve_bm25",
        "retrieve_web": "retrieve_web",
        "direct_answer": "direct_answer",
    }
)

# Retrieval branches converge on the generator.
graph.add_edge("retrieve_vector", "generate")
graph.add_edge("retrieve_bm25", "generate")
graph.add_edge("retrieve_web", "generate")

# Generator and direct both end.
graph.add_edge("generate", END)
graph.add_edge("direct_answer", END)

app = graph.compile()
print("[graph] strategy-selection graph compiled")


### 11. Visualize the Graph


In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not render graph image: {e}")
    print("Graph structure:")
    print("  START -> classify")
    print("    classify -> retrieve_vector   -> generate -> END")
    print("    classify -> retrieve_bm25     -> generate -> END")
    print("    classify -> retrieve_web      -> generate -> END")
    print("    classify -> direct_answer                  -> END")


### 12. Test: Factual Query (should route to BM25)


In [ ]:
print("=" * 60)
print("TEST 1: Factual query -> expect bm25 strategy")
print("=" * 60)
result = app.invoke({
    "query": "What did the Cheshire Cat say to Alice?",
    "classification": "",
    "strategy": "",
    "context": "",
    "answer": "",
    "route_log": [],
})
print(f"\nClassification: {result['classification']}")
print(f"Strategy used: {result['strategy']}")
print(f"Answer: {result['answer'][:300]}")
print(f"Route log: {result['route_log']}")


### 13. Test: Analytical Query (should route to Vector)


In [ ]:
print("\n" + "=" * 60)
print("TEST 2: Analytical query -> expect vector strategy")
print("=" * 60)
result = app.invoke({
    "query": "Why does the author use nonsense poetry to convey deeper meaning?",
    "classification": "",
    "strategy": "",
    "context": "",
    "answer": "",
    "route_log": [],
})
print(f"\nClassification: {result['classification']}")
print(f"Strategy used: {result['strategy']}")
print(f"Answer: {result['answer'][:300]}")
print(f"Route log: {result['route_log']}")


### 14. Test: Conversational Query (should route to Direct)


In [ ]:
print("\n" + "=" * 60)
print("TEST 3: Conversational query -> expect direct strategy")
print("=" * 60)
result = app.invoke({
    "query": "Hello! Can you tell me a fun fact about Alice in Wonderland?",
    "classification": "",
    "strategy": "",
    "context": "",
    "answer": "",
    "route_log": [],
})
print(f"\nClassification: {result['classification']}")
print(f"Strategy used: {result['strategy']}")
print(f"Answer: {result['answer'][:300]}")
print(f"Route log: {result['route_log']}")


### 15. Test: Out-of-Scope Query (should route to Web)


In [ ]:
print("\n" + "=" * 60)
print("TEST 4: Out-of-scope query -> expect web strategy")
print("=" * 60)
result = app.invoke({
    "query": "What is the latest news about AI regulation in the EU?",
    "classification": "",
    "strategy": "",
    "context": "",
    "answer": "",
    "route_log": [],
})
print(f"\nClassification: {result['classification']}")
print(f"Strategy used: {result['strategy']}")
print(f"Answer: {result['answer'][:300]}")
print(f"Route log: {result['route_log']}")


### 16. Inspect Graph Structure


In [ ]:
print("=== Graph Structure ===")
g = app.get_graph()
print(f"Nodes: {list(g.nodes.keys())}")
print("Edges:")
for edge in g.edges:
    print(f"  {edge.source} -> {edge.target}")


### Summary
